# employee data analysis

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import missingno as msno
from sklearn.impute import KNNImputer 
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
CUR_DIR = Path.cwd().resolve()


### 1.1 load dataset

In [ ]:
df_raw = pd.read_csv(CUR_DIR / 'dataset/employee_2023.csv', sep=',', index_col=None)

## 2. Initial Data Exploration - Getting to Know Our Data

Before diving into cleaning, we need to understand what we're working with.


In [ ]:
print("=" * 80)
print(" INITIAL DATA EXPLORATION")
print("=" * 80)

print(f"   Features: {df_raw.dtypes}")
print(f"\n First look at the data:")
display(df_raw.head())

print(f"\n Dataset Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"\n Memory Usage: {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")

# Data types and completeness
print("\n Data Types and Completeness:")
dtype_summary = pd.DataFrame({
    'Data Type': df_raw.dtypes,
    'Non-Null Count': df_raw.notnull().sum(),
    'Null Count': df_raw.isnull().sum(),
    'Null %': (df_raw.isnull().sum() / len(df_raw) * 100).round(1),
    'Unique Values': df_raw.nunique()
})

# Numerical columns overview
print("\n Quick Statistical Summary (Numeric Columns):")
display(df_raw.describe().round(2))

# Categorical columns overview
print("\n Categorical Columns Overview:")
cat_cols = df_raw.select_dtypes(include=['object']).columns
for col in cat_cols:
    if col != 'name':
        print(f"\n{col}:")
        print(df_raw[col].value_counts().to_string()) # todo: hire_date too long (shwo first couple rows) 


## 3. Data Quality Assessment - Systematic Problem Finding
Now we audit every column against rules.
### Good data rules:
1. COMPLETENESS - All required values are present
2. VALIDITY - Values follow business rules
3. CONSISTENCY - Values make sense together
4. UNIQUENESS - No unwanted duplicates
5. ACCURACY - Values reflect reality (hard to verify)

In [ ]:
class DataQualityAssessor:
    """
    data quality assessment.
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.rules = self._define_rules()
        self.quality_report = {}
        
    def _define_rules(self) -> dict:
        """Define business rules that data must satisfy."""
        return {
            "Age must be positive (18-70)": 
                (self.df["age"] > 0) & (self.df["age"] <= 70),
            "Salary must be reasonable ($20k-$500k)": 
                (self.df["salary"] > 20000) & (self.df["salary"] < 500000),
            "Performance score must be 0-100": 
                (self.df["performance_score"] >= 0) & (self.df["performance_score"] <= 100),
            "Projects completed must be non-negative": 
                self.df["projects_completed"] >= 0,
            "Overtime hours must be non-negative": 
                self.df["overtime_hours"] >= 0,
            "Tenure must be non-negative": 
                self.df["tenure_years"] >= 0,
            "Team size must be positive": 
                self.df["team_size"] > 0,
            "Hire date cannot be in the future": 
                pd.to_datetime(self.df['hire_date']).dt.date <= datetime.now().date(),
            "Satisfaction must be a valid category": 
                self.df["satisfaction"].isin(['Very Low', 'Low', 'Medium', 'High', 'Very High'])
        }
    
    def assess(self) -> dict:
        """Run quality assessment."""
        
        # Check completeness
        completeness = pd.DataFrame({
            'Column': self.df.columns,
            'Missing': self.df.isnull().sum(),
            'Complete %': ((1 - self.df.isnull().sum() / len(self.df)) * 100).round(1)
        }).sort_values('Missing', ascending=False)
        
        # Check validity (business rules)
        violations = {}
        for rule, condition in self.rules.items():
            violations[rule] = (~condition.fillna(True)).sum()
        
        # Check uniqueness
        duplicates = self.df.duplicated().sum()
        id_duplicates = self.df['employee_id'].duplicated().sum()
        
        # quality score
        total_checks = len(self.df) * len(self.rules)
        total_violations = sum(violations.values())
        quality_score = (1 - total_violations / total_checks) * 100
        
        self.quality_report = {
            'completeness': completeness,
            'violations': pd.Series(violations),
            'duplicate_rows': duplicates,
            'duplicate_ids': id_duplicates,
            'quality_score': quality_score
        }
        
        return self.quality_report
    
    def print_report(self):
        """Print a quality report."""
        if not self.quality_report:
            self.assess()
        
        r = self.quality_report
        
        print("=" * 80)
        print(" DATA QUALITY ASSESSMENT REPORT")
        print("=" * 80)
        
        print(f"\n Overall Quality Score: {r['quality_score']:.1f}%")
        
        print(f"\n Completeness Issues:")
        missing = r['completeness'][r['completeness']['Missing'] > 0]
        if len(missing) > 0:
            display(missing)
        else:
            print(" No missing values found")
        
        print(f"\n Business Rule Violations:")
        violations = r['violations'][r['violations'] > 0]
        if len(violations) > 0:
            for rule, count in violations.items():
                pct = (count / len(self.df)) * 100
                print(f"   • {rule}: {count} violations ({pct:.1f}%)")
        else:
            print(" No rule violations found")
        
        print(f"\n Uniqueness:")
        print(f"   - Duplicate rows: {r['duplicate_rows']}")
        print(f"   - Duplicate employee IDs: {r['duplicate_ids']}")

assessor = DataQualityAssessor(df_raw)
assessor.assess()
assessor.print_report()

## 4. Missing Data Analysis - Understanding Why Data is Missing
key assertions: 
- Before we can fix missing data, we need to understand **WHY** it's missing.
- Different types of missingness require **different handling strategies**.


In [ ]:
print("=" * 80)
print(" MISSING DATA ANALYSIS")
print("=" * 80)

# Visualize missing patterns
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

msno.matrix(df_raw, ax=axes[0], fontsize=8)
axes[0].set_title('Missing Data Matrix\n(White lines = missing values)', 
                  fontsize=12, fontweight='bold')

msno.heatmap(df_raw, ax=axes[1])
axes[1].set_title('Correlation of Missingness\n(Higher numbers = missing together)', 
                  fontsize=12, fontweight='bold')

msno.bar(df_raw, ax=axes[2])
axes[2].set_title('Data Completeness by Column', 
                  fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Analyze MAR patterns - is salary missingness related to department?
print("\n MAR Analysis: Is salary missingness related to department?")
missing_by_dept = pd.crosstab(
    df_raw['department'],
    df_raw['salary'].isna(),
    normalize='index'
)
missing_by_dept.columns = ['Has Salary', 'Missing Salary']
display(missing_by_dept)

# Analyze MNAR patterns - is overtime missingness related to overtime values?
print("\n MNAR Analysis: Is overtime missingness related to overtime values?")
ot_median = df_raw['overtime_hours'].median()
high_ot = df_raw[df_raw['overtime_hours'] > ot_median]
low_ot = df_raw[df_raw['overtime_hours'] <= ot_median]
print(f"   High overtime missing rate: {high_ot['overtime_hours'].isna().mean()*100:.1f}%")
print(f"   Low overtime missing rate: {low_ot['overtime_hours'].isna().mean()*100:.1f}%")
print(f"   Difference: This suggests MNAR - missingness depends on the value itself")

# Visualize the three mechanisms
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MCAR
mcar_cols = ['projects_completed', 'training_completed', 'team_size']
mcar_missing = df_raw[mcar_cols].isna().sum()
axes[0].bar(mcar_cols, mcar_missing, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('MCAR: Random Missing\n(No pattern, ~5% each)', fontweight='bold')
axes[0].set_ylabel('Missing Count')

# MAR
dept_missing = df_raw.groupby('department')['salary'].apply(lambda x: x.isna().mean() * 100)
axes[1].bar(dept_missing.index, dept_missing.values, color='#e67e22')
axes[1].set_title('MAR: Salary Missing by Dept\n(HR has more missing)', fontweight='bold')
axes[1].set_ylabel('Missing %')
axes[1].tick_params(axis='x', rotation=45)

# MNAR
axes[2].bar(['Low OT\n(≤ median)', 'High OT\n(> median)'],
           [low_ot['overtime_hours'].isna().mean()*100,
            high_ot['overtime_hours'].isna().mean()*100],
           color=['#2ecc71', '#e74c3c'])
axes[2].set_title('MNAR: OT Missing by Value\n(High OT = more missing)', fontweight='bold')
axes[2].set_ylabel('Missing %')

plt.suptitle('Three Types of Missing Data Mechanisms', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Data Cleaning - Fixing What's Broken

- Fix the issues we found. 

In [ ]:
class DataCleaner:
    """
    Systematic data cleaning with full traceability.
    """
    
    def __init__(self):
        self.cleaning_log = []
    
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Execute complete cleaning pipeline."""
        df_clean = df.copy()
        print(" Starting data cleaning...\n")
        
        # Remove duplicates
        df_clean = self._remove_duplicates(df_clean)
        
        # Fix data types
        df_clean = self._fix_dates(df_clean)
        
        # Fix categorical typos
        df_clean = self._fix_categories(df_clean)
        
        # Handle impossible values
        df_clean = self._fix_impossible_values(df_clean)
        
        # Handle outliers
        df_clean = self._handle_outliers(df_clean)
        
        print(f"\n Cleaning complete. {len(self.cleaning_log)} actions performed.")
        return df_clean
    
    def _remove_duplicates(self, df):
        """Remove exact and near-duplicate records."""
        before = len(df)
        
        # Remove exact duplicates
        df = df.drop_duplicates()
        exact_dupes = before - len(df)
        
        # Remove duplicate employee IDs (keep most recent record)
        if df['employee_id'].duplicated().any():
            df = df.sort_values('hire_date').drop_duplicates('employee_id', keep='last')
        
        after = len(df)
        self._log(f"Removed {exact_dupes} exact duplicate rows")
        if before - after > exact_dupes:
            self._log(f"Removed {before - after - exact_dupes} duplicate employee IDs")
        
        return df.reset_index(drop=True)
    
    def _fix_dates(self, df):
        """Fix impossible dates."""
        # Convert to datetime
        df['hire_date'] = pd.to_datetime(df['hire_date'])
        
        # Fix future dates
        future = df['hire_date'] > pd.Timestamp.now()
        if future.sum() > 0:
            df.loc[future, 'hire_date'] = df.loc[future, 'hire_date'].apply(
                lambda d: d.replace(year=d.year - 1)
            )
            self._log(f"Corrected {future.sum()} future hire dates")
        
        return df
    
    def _fix_categories(self, df):
        """Standardize categorical values."""
        # Fix satisfaction typos
        satisfaction_map = {
            'vlow': 'Very Low', 'Vlow': 'Very Low',
            'low ': 'Low', 'LOW': 'Low',
            'medium': 'Medium',
            'high': 'High', 'HIGH': 'High',
            'very high': 'Very High'
        }
        
        if 'satisfaction' in df.columns:
            original_unique = df['satisfaction'].nunique()
            df['satisfaction'] = df['satisfaction'].str.strip().str.title()
            df['satisfaction'] = df['satisfaction'].replace(satisfaction_map)
            new_unique = df['satisfaction'].nunique()
            
            if original_unique != new_unique:
                self._log(f"Standardized satisfaction values from {original_unique} to {new_unique} unique categories")
        
        return df
    
    def _fix_impossible_values(self, df):
        """Fix values that can't possibly be correct."""
        
        # Negative ages -> likely data entry error, use absolute value
        neg_ages = (df['age'] < 0).sum()
        if neg_ages > 0:
            df.loc[df['age'] < 0, 'age'] = df.loc[df['age'] < 0, 'age'].abs()
            self._log(f"Corrected {neg_ages} negative ages to positive")
        
        # Ages over 70 -> flag but keep (could be legitimate)
        old_ages = (df['age'] > 70).sum()
        if old_ages > 0:
            df['age_flag'] = df['age'] > 70
            self._log(f"Flagged {old_ages} ages over 70 for review")
        
        return df
    
    def _handle_outliers(self, df):
        """Handle outliers carefully - flag rather than delete."""
        
        # Salary outliers - flag and cap for analysis
        salary_bounds = df['salary'].quantile([0.01, 0.99]).values
        salary_outliers = (df['salary'] < salary_bounds[0]) | (df['salary'] > salary_bounds[1])
        
        if salary_outliers.sum() > 0:
            df['salary_outlier'] = salary_outliers
            df['salary_original'] = df['salary'].copy()
            df['salary'] = df['salary'].clip(*salary_bounds)
            self._log(f"Flagged and capped {salary_outliers.sum()} salary outliers")
        
        # Overtime outliers
        ot_bounds = df['overtime_hours'].quantile([0.01, 0.99]).values
        ot_outliers = (df['overtime_hours'] < ot_bounds[0]) | (df['overtime_hours'] > ot_bounds[1])
        
        if ot_outliers.sum() > 0:
            df['ot_outlier'] = ot_outliers
            df['overtime_hours'] = df['overtime_hours'].clip(*ot_bounds)
            self._log(f"Flagged and capped {ot_outliers.sum()} overtime outliers")
        
        return df
    
    def _log(self, message):
        """Record cleaning action for audit trail."""
        self.cleaning_log.append(message)
        print(f"   + {message}")

cleaner = DataCleaner()
df_clean = cleaner.clean(df_raw)

print("\n Post-Cleaning Quality Check:")
final_assessor = DataQualityAssessor(df_clean)
final_assessor.assess()
final_assessor.print_report()


## 6. Missing Value Imputation
key assertion:
- Now we fill in the remaining missing values. We use different strategies
- for different types of data because one size doesn't fit all.


In [ ]:
print("=" * 80)
print(" MISSING VALUE IMPUTATION")
print("=" * 80)

def plot_improvements(k_values, mse_scores, optimal_k=None):
    """Plot elbow curve with selected `k`"""
    
    global plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(k_values, mse_scores, marker='o', linewidth=2, markersize=8)
    axes[0].set_xlabel('Number of Neighbors (k)', fontsize=12)
    axes[0].set_ylabel('Mean Squared Error', fontsize=12)
    axes[0].set_title('Elbow Method: Finding Optimal k for KNN Imputation', 
                     fontsize=13, fontweight='bold')
    if optimal_k is not None:
        axes[0].axvline(x=optimal_k, color='red', linestyle='--', alpha=0.5, label=f'Selected k={optimal_k}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Show improvement
    improvement = np.abs(np.diff(mse_scores) / mse_scores[:-1] * 100)
    axes[1].bar(range(2, 16), improvement)
    axes[1].set_xlabel('Number of Neighbors (k)', fontsize=12)
    axes[1].set_ylabel('Improvement from Previous k (%)', fontsize=12)
    axes[1].set_title('Marginal Improvement per Additional Neighbor', 
                     fontsize=13, fontweight='bold')
    axes[1].axhline(y=5, color='red', linestyle='--', alpha=0.5, 
                   label='5% threshold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Separate numeric and categorical columns
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['employee_id', 'age_flag', 
                                                       'salary_outlier', 'ot_outlier']]
cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"\n Numeric columns to impute: {numeric_cols}")
print(f" Categorical columns to impute: {cat_cols}")

# First, let's find optimal k for KNN imputation
print("\n Finding optimal k for KNN imputation...")

# Use salary as test case (has clear missing values)
salary_complete = df_clean[['salary', 'age', 'tenure_years', 'performance_score']].dropna()
if len(salary_complete) > 100:
    # Create artificial missing values for testing
    np.random.seed(42)
    mask = np.random.rand(len(salary_complete)) < 0.2
    salary_missing = salary_complete.copy()
    salary_missing.loc[mask, 'salary'] = np.nan
    
    # Test different k values
    k_values = range(1, 16)
    mse_scores = []
    
    for k in k_values:
        imputer = KNNImputer(n_neighbors=k)
        imputed = imputer.fit_transform(salary_missing)
        mse = mean_squared_error(
            salary_complete['salary'][mask],
            imputed[mask, 0]
        )
        mse_scores.append(mse)
    plot_improvements(k_values, mse_scores) # plot improvements

    optimal_k = int(input('choose number of clusters (`k` value):')) # `4`` is optimal ask to choose `k` value (number of clusters)
    print(f"   Selected k = {optimal_k} (balance between accuracy and overfitting)")
    plot_improvements(k_values, mse_scores, optimal_k) # plot improvements

# Apply imputation
print("\n Applying KNN imputation to numeric columns...")
imputer_knn = KNNImputer(n_neighbors=optimal_k)
df_clean[numeric_cols] = imputer_knn.fit_transform(df_clean[numeric_cols])
print(f"    Imputed {len(numeric_cols)} numeric columns")

# Categorical columns - use mode (most frequent value)
print("\n Applying mode imputation to categorical columns...")
cat_missing = df_clean[cat_cols].isnull().sum()
cat_missing = cat_missing[cat_missing > 0]

for col in cat_missing.index:
    mode_val = df_clean[col].mode()[0]
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"   + Imputed {col} with mode: '{mode_val}'")

print(f"\n Imputation complete! Remaining missing values: {df_clean.isnull().sum().sum()}")
